In [6]:
import pandas as pd

# 1️⃣ County 단위 Population 데이터 불러오기
df_population = pd.read_csv("Total_Population.csv", low_memory=False)

# 2️⃣ County 단위 Educational Attainment 데이터 불러오기
df_edu_county = pd.read_csv("Educational_Attainment_US_County.csv", low_memory=False)

# 3️⃣ State Area 데이터 불러오기
df_area = pd.read_csv("State_Area.csv", low_memory=False)

# --- 인구 밀도 계산 (Population Density) ---

# County → State 이름 추출
df_population = df_population[~df_population['NAME'].str.contains('Geographic Area Name', na=False)].copy()
df_population['State'] = df_population['NAME'].str.extract(r',\s*([\w\s]+)$')[0]

# 숫자 변환
df_population['Population'] = pd.to_numeric(df_population['B01003_001E'], errors='coerce')

# State별 총 인구 집계
state_population = df_population.groupby('State')['Population'].sum().reset_index()

# State별 면적 정리
df_area.rename(columns={'State': 'State', 'Total Area (sq mi)': 'Area_sqmi'}, inplace=True)

# Population과 Area 병합
df_density = pd.merge(state_population, df_area[['State', 'Area_sqmi']], on='State', how='inner')

# Population Density 계산
df_density['Population_Density'] = df_density['Population'] / df_density['Area_sqmi']

# --- 교육 수준 계산 (College Educated Share) ---

# Educational Data 클린
df_edu_county = df_edu_county[~df_edu_county['NAME'].str.contains('Geographic Area Name', na=False)].copy()
df_edu_county['State'] = df_edu_county['NAME'].str.extract(r',\s*([\w\s]+)$')[0]

# 숫자 변환
cols_to_numeric = ['B15003_001E', 'B15003_022E', 'B15003_023E', 'B15003_024E', 'B15003_025E']
for col in cols_to_numeric:
    df_edu_county[col] = pd.to_numeric(df_edu_county[col], errors='coerce')

# County별 College Educated Share 계산
df_edu_county['College_Educated_Share'] = (
    df_edu_county[['B15003_022E', 'B15003_023E', 'B15003_024E', 'B15003_025E']].sum(axis=1) /
    df_edu_county['B15003_001E']
)

# State별 평균 College Educated Share 계산
state_edu = df_edu_county.groupby('State')['College_Educated_Share'].mean().reset_index()

# --- X_it 최종 병합 ---

X_it = pd.merge(df_density[['State', 'Population_Density']], state_edu, on='State', how='inner')

# 결과 출력
print(X_it.head(10))

# 필요하면 저장
# X_it.to_csv("X_it_controls.csv", index=False)


# 4️⃣ State 단위 산업 구조 데이터 불러오기
df_industry = pd.read_csv("State-Industry_Shares__2020-2022_.csv", low_memory=False)

# 산업 구조 데이터 확인
print(df_industry.head())

# 산업 구조 데이터 클린업 (State 컬럼명 확인 후 맞춰주기)
if 'State' not in df_industry.columns:
    # 다른 이름으로 되어 있으면 수정 (예: 'state', 'STATE', 'Area Name' 등)
    df_industry.rename(columns={'Area Name': 'State'}, inplace=True)

# 필요한 컬럼만 추출 (State + 산업별 비중 컬럼)
industry_columns = [col for col in df_industry.columns if col != 'State']
df_industry = df_industry[['State'] + industry_columns]

# --- X_it + 산업 구조 병합 ---

# 기존 X_it와 병합
X_it_full = pd.merge(X_it, df_industry, on='State', how='inner')

# 결과 확인
print(X_it_full.head(10))

# 필요하면 저장
# X_it_full.to_csv("X_it_full_controls.csv", index=False)


                  State  Population_Density  College_Educated_Share
0               Alabama           72.804445                0.285865
1                Alaska            0.745666                0.328481
2               Arizona           63.712422                0.260057
3              Arkansas           33.716843                0.296515
4            California          235.669617                0.329415
5              Colorado           48.739572                0.466790
6           Connecticut          652.566480                0.413187
7              Delaware          414.580153                0.339129
8  District of Columbia         9984.882353                0.659442
9               Florida          333.783616                0.337228
           State  Year  Construction_Share  EducationHealth_Share  \
0  United States  2020            0.069124               0.031237   
1  United States  2021            0.067229               0.043225   
2  United States  2022            0.085925   

In [7]:
import pandas as pd

# --- CBP 데이터 불러오기 ---

# 파일 경로
files = [
    "CBP2020.CB2000CBP-Data.csv",
    "CBP2021.CB2100CBP-Data.csv",
    "CBP2022.CB2200CBP-Data.csv"
]


cbp_data = []
for file in files:
    df = pd.read_csv(file, low_memory=False)
    df['Year'] = int(file.split('CBP')[1][:4])
    cbp_data.append(df)

# 하나로 합치기
df_cbp = pd.concat(cbp_data, ignore_index=True)

# --- State 이름 추출하기 ---

# NAME에서 주 이름 뽑기
df_cbp['State'] = df_cbp['NAME'].str.extract(r',\s*([\w\s]+)$')[0]

# --- 필요한 컬럼만 정리 ---
columns_to_keep = ['State', 'NAICS2017', 'EMP', 'Year']
df_cbp = df_cbp[columns_to_keep]

# NAICS 컬럼 이름 통일
df_cbp.rename(columns={'NAICS2017': 'NAICS'}, inplace=True)

# EMP 숫자 변환
df_cbp['EMP'] = pd.to_numeric(df_cbp['EMP'], errors='coerce')

# 2022년 데이터만 선택
df_cbp_2022 = df_cbp[df_cbp['Year'] == 2022].copy()

# --- State별 산업별 고용자 수 합계 ---

# State-Industry 레벨로 합산
state_industry = df_cbp_2022.groupby(['State', 'NAICS'])['EMP'].sum().reset_index()

# State별 전체 고용자 수
state_total_emp = state_industry.groupby('State')['EMP'].sum().reset_index()
state_total_emp.rename(columns={'EMP': 'Total_EMP'}, inplace=True)

# 병합 후 산업별 고용 비율 계산
state_industry = pd.merge(state_industry, state_total_emp, on='State', how='left')
state_industry['Industry_Share'] = state_industry['EMP'] / state_industry['Total_EMP']

# Pivot Table로 변환 (State x NAICS 매트릭스)
df_industry_share = state_industry.pivot_table(index='State', columns='NAICS', values='Industry_Share', fill_value=0).reset_index()

# 컬럼 이름 정리
df_industry_share.columns = [str(col) if col != 'State' else col for col in df_industry_share.columns]

# 결과 출력
print(df_industry_share.head())

# 필요하면 저장
# df_industry_share.to_csv("State_Industry_Share_2022.csv", index=False)

        State        00        11        21        22        23     31-33  \
0     Alabama  0.500404  0.001598  0.001303  0.003788  0.029325  0.078668   
1      Alaska  0.501733  0.001285  0.016241  0.003893  0.032241  0.020623   
2     Arizona  0.500136  0.000357  0.002018  0.002415  0.037360  0.032460   
3    Arkansas  0.500790  0.002248  0.001204  0.003790  0.025699  0.077403   
4  California  0.500011  0.001113  0.000477  0.002187  0.029134  0.038742   

         42     44-45     48-49  ...        53        54        55        56  \
0  0.021819  0.068437  0.020952  ...  0.007488  0.032261  0.006274  0.031978   
1  0.015602  0.066856  0.039878  ...  0.009333  0.038411  0.009094  0.034409   
2  0.018435  0.065598  0.024030  ...  0.011357  0.034477  0.012694  0.034792   
3  0.022552  0.068765  0.029948  ...  0.006470  0.018137  0.016621  0.023306   
4  0.026464  0.056107  0.023653  ...  0.010849  0.045707  0.012251  0.030708   

         61        62        71        72        81     

In [8]:
# 기존 X_it (인구밀도 + 교육 수준)과 병합
X_it_final = pd.merge(X_it, df_industry_share, on='State', how='inner')

# 결과 출력
print(X_it_final.head())

# 필요하면 저장
# X_it_final.to_csv("X_it_final_with_industry.csv", index=False)


        State  Population_Density  College_Educated_Share        00        11  \
0     Alabama           72.804445                0.285865  0.500404  0.001598   
1      Alaska            0.745666                0.328481  0.501733  0.001285   
2     Arizona           63.712422                0.260057  0.500136  0.000357   
3    Arkansas           33.716843                0.296515  0.500790  0.002248   
4  California          235.669617                0.329415  0.500011  0.001113   

         21        22        23     31-33        42  ...        53        54  \
0  0.001303  0.003788  0.029325  0.078668  0.021819  ...  0.007488  0.032261   
1  0.016241  0.003893  0.032241  0.020623  0.015602  ...  0.009333  0.038411   
2  0.002018  0.002415  0.037360  0.032460  0.018435  ...  0.011357  0.034477   
3  0.001204  0.003790  0.025699  0.077403  0.022552  ...  0.006470  0.018137   
4  0.000477  0.002187  0.029134  0.038742  0.026464  ...  0.010849  0.045707   

         55        56        61 

In [9]:
# 1. State 알파벳순 정렬
X_it_final_sorted = X_it_final.sort_values(by='State').reset_index(drop=True)

# 2. 컬럼 순서 정리
ordered_cols = ['State', 'Population_Density', 'College_Educated_Share'] + \
    [col for col in X_it_final.columns if col not in ['State', 'Population_Density', 'College_Educated_Share']]

X_it_final_sorted = X_it_final_sorted[ordered_cols]

# 3. 결과 미리보기
print(X_it_final_sorted)

# 4. 저장 (선택사항)
# X_it_final_sorted.to_csv("X_it_final_with_industry_sorted.csv", index=False)


                   State  Population_Density  College_Educated_Share  \
0                Alabama           72.804445                0.285865   
1                 Alaska            0.745666                0.328481   
2                Arizona           63.712422                0.260057   
3               Arkansas           33.716843                0.296515   
4             California          235.669617                0.329415   
5               Colorado           48.739572                0.466790   
6            Connecticut          652.566480                0.413187   
7               Delaware          414.580153                0.339129   
8   District of Columbia         9984.882353                0.659442   
9                Florida          333.783616                0.337228   
10               Georgia          145.090332                0.322504   
11                Hawaii          131.276802                0.332773   
12                 Idaho           15.367888                0.31

In [17]:
# X_it_final_sorted를 CSV 파일로 저장
X_it_final_sorted.to_csv('X_it_for_regression.csv', index=False)

print("✅ 저장 완료! 파일명: X_it_for_regression.csv")


✅ 저장 완료! 파일명: X_it_for_regression.csv
